# CatBreedID – Khảo sát dữ liệu (EDA)

Số liệu và hình cho **Chương 3, phần dữ liệu**, theo bản bàn giao `BanGiao_EDA_Tai_2026-11-16.md`.

**Quy tắc tuân thủ trong notebook này**
1. Oxford trước khi khóa: chỉ dùng split **trainval**. Danh sách và ảnh test chỉ được đọc trong mục D, và mục D chỉ chạy khi `ALLOW_TEST = True` (đổi sau 19:10 khi Hậu báo khóa).
2. Không dùng mô hình: không checkpoint, không điểm số, không dự đoán, không Grad-CAM. Chỉ dùng ảnh, nhãn, trimap, hộp bao.
3. Dữ liệu giải nén từ archive chính thức (MD5 đã kiểm, xem mục 0). FeliBreed là bản đã đưa lên `roonyvtt/catbreed-felibreed-unseen`.
4. Chạy bằng CPU trên máy riêng.
5. Bảng lưu ở `tables/*.csv`, hình ở `figures/*.png` (300 dpi, chữ đen, không tiêu đề trên hình; tiêu đề viết trong báo cáo).
6. Mỗi nhận xét tách **Quan sát** (số liệu) và **Suy đoán** (ảnh hưởng có thể tới mô hình).

Thư mục dữ liệu mặc định là `../../eda_data` (tính từ thư mục `eda/`), đổi được bằng biến môi trường `EDA_DATA`. Thư mục này không nằm trong repo.

```
eda_data/
  oxford/images, oxford/annotations   giải nén images.tar.gz, annotations.tar.gz
  dtd/dtd/images                      giải nén dtd-r1.0.1.tar.gz
  felibreed/                          bản roonyvtt/catbreed-felibreed-unseen (FeliBreed/, manifest.csv, README.md)
  archives/                           3 archive gốc, để kiểm MD5
  splits/                             fixed_split.csv, data_audit.json do notebook sinh ra
```

## 0. Cấu hình, kiểm tra nguồn dữ liệu

In [1]:
import hashlib
import json
import os
import warnings
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.patches import Rectangle
from PIL import Image
from scipy import stats

Image.MAX_IMAGE_PIXELS = None
pd.set_option("display.max_columns", 40, "display.width", 200, "display.max_rows", 80)

ROOT = Path(os.environ.get("EDA_DATA", "../../eda_data")).resolve()
OXFORD, DTD, FELI, SPLITS = ROOT / "oxford", ROOT / "dtd" / "dtd" / "images", ROOT / "felibreed", ROOT / "splits"
TABLES, FIGURES = Path("tables"), Path("figures")
TABLES.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

ALLOW_TEST = False  # Chỉ đổi thành True sau 19:10, khi Hậu báo đã khóa cấu hình
RESIZE, CROP = 256, 224  # Resize cạnh ngắn 256 rồi CenterCrop 224, như notebook
SEED = 0
EXPECTED_MD5 = {
    "images.tar.gz": "5c4f3ee8e5d25df40f4fd59a7f44e54c",
    "annotations.tar.gz": "95a8c909bbe2e81eed6a22bccdf3f68f",
    "dtd-r1.0.1.tar.gz": "fff73e5086ae6bdbea199a49dfb8a4c1",
}
EXPECTED_SPLIT_ID = "3b283d422973d497d268d935ef0fe4a76bbd2d10f4b5efdef20a8d859949cfcb"
EXPECTED_PROBE_ID = "47a50e31b6f609c32aaadb30faa60614bcf5ac518c71fbde68ad8fe045947daf"
PROBE_PER_BREED, PROBE_SEED = 20, 42
CAT_BREEDS = ["Abyssinian", "Bengal", "Birman", "Bombay", "British_Shorthair", "Egyptian_Mau",
              "Maine_Coon", "Persian", "Ragdoll", "Russian_Blue", "Siamese", "Sphynx"]
DATASETS = ["Mèo trainval (ID)", "Chó trainval", "DTD", "FeliBreed"]
SHORT = {"Mèo trainval (ID)": "Mèo ID", "Chó trainval": "Chó", "DTD": "DTD", "FeliBreed": "FeliBreed"}
COLORS = {"Mèo trainval (ID)": "#2a6fdb", "Chó trainval": "#e08a1e", "DTD": "#6b7785", "FeliBreed": "#2b9e8f"}
SPLIT_COLORS = {"train": "#2a6fdb", "val": "#e08a1e"}

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 300, "font.size": 10, "font.family": "DejaVu Sans",
    "text.color": "black", "axes.labelcolor": "black", "xtick.color": "black", "ytick.color": "black",
    "axes.edgecolor": "black", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#dddddd", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "legend.frameon": False,
})


def save(fig, name):
    """Lưu 300 dpi, không tiêu đề trên hình; tiêu đề viết trong báo cáo."""
    fig.savefig(FIGURES / f"{name}.png", dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    text = FIGURE_CAPTIONS.get(name) or OOD_GRID_CAPTIONS.get(name.removeprefix("A4_").removesuffix("_12_random"))
    if text:
        display(Markdown(f"<p style='text-align:center'><b>{caption_text('Hình', name, text)}</b></p>"))


def save_table(frame, name, index=False):
    frame.to_csv(TABLES / f"{name}.csv", index=index, encoding="utf-8-sig")
    if TABLE_CAPTIONS.get(name):
        show_table(frame, name, index=index)




FIGURE_CAPTIONS, TABLE_CAPTIONS, CAPTION_LOG = {'A3_short_side_hist': 'Phân bố cạnh ngắn của ảnh trong bốn tập (thang log); nét chấm: 224 pixel, nét đứt: 256 pixel', 'A3_aspect_ratio_hist': 'Phân bố tỷ lệ khung rộng/cao của ảnh trong bốn tập; nét đứt: ảnh vuông', 'A4_cat_breeds_6_each': 'Sáu ảnh ngẫu nhiên (seed 0) của mỗi giống mèo, trainval', 'A4_{name}_12_random': None, 'B1_breed_counts_train_val': 'Số ảnh mỗi giống mèo trong train và validation', 'B2_near_duplicate_examples': 'Các cặp ảnh gần trùng trong mèo trainval theo khoảng cách Hamming của phash', 'B3_felibreed_internal_examples': 'Các cặp ảnh gần trùng trong nội bộ FeliBreed', 'B3_felibreed_vs_oxford_examples': 'Các cặp ảnh gần trùng giữa FeliBreed và Oxford trainval', 'C1_foreground_fraction_hist': 'Phân bố tỷ lệ diện tích con vật trên ảnh theo trimap, mèo trainval', 'C1_foreground_centre_heatmap': 'Vị trí tâm con vật trong ảnh theo từng giống, mèo trainval; dấu + là tâm ảnh', 'C2_crop_retention_hist': 'Tỷ lệ pixel con vật còn lại sau Resize 256 → CenterCrop 224, mèo trainval', 'C2_crop_retention_worst': 'Tám ảnh mất nhiều tiền cảnh nhất khi cắt giữa; khung đỏ là phần còn lại', 'C3_head_box_hist': 'Kích thước tương đối của hộp đầu (trái) và phần hộp đầu còn lại sau cắt giữa (phải)', 'C4_image_stats_by_breed': 'Độ sáng, độ tương phản, độ nét và độ bão hòa theo giống, mèo trainval', 'C4_image_stats_by_dataset': 'Độ sáng, độ tương phản, độ nét và độ bão hòa theo tập dữ liệu'}, {'00_archive_md5': 'Kiểm tra MD5 của ba archive gốc', 'A1_dataset_description': 'Mô tả bốn tập dữ liệu', 'A2_image_quality': 'Ảnh lỗi, định dạng thật, chế độ màu và cờ xoay EXIF', 'A2_read_errors': 'Danh sách ảnh lỗi khi đọc', 'A2_non_rgb_exif_or_format_mismatch': 'Ảnh không phải RGB, có thẻ EXIF Orientation khác 1 hoặc định dạng khác đuôi tệp', 'A3_image_size_summary': 'Tóm tắt kích thước ảnh của bốn tập', 'B1_breed_counts_train_val': 'Số ảnh mỗi giống mèo trong train và validation', 'B1_imbalance_ratio': 'Mức mất cân bằng giữa các giống', 'near_duplicates_trainval': 'Danh sách cặp ảnh gần trùng trong mèo trainval (Hamming ≤ 8)', 'B2_near_duplicate_threshold_counts': 'Số cặp gần trùng trong mèo trainval theo ngưỡng Hamming', 'B3_felibreed_near_duplicate_counts': 'Số cặp gần trùng của FeliBreed theo ngưỡng Hamming', 'near_duplicates_felibreed': 'Danh sách cặp ảnh gần trùng liên quan tới FeliBreed (Hamming ≤ 8)', 'B4_felibreed_breed_names': 'Thư mục giống trong FeliBreed và quyết định giữ hoặc loại', 'C1_trimap_foreground_per_image': None, 'C1_trimap_foreground_summary': 'Diện tích và độ lệch tâm của con vật theo trimap, mèo trainval', 'crop_retention_trainval': None, 'C2_crop_retention_summary': 'Tiền cảnh còn lại sau Resize 256 → CenterCrop 224', 'C2_crop_retention_by_breed': 'Tiền cảnh còn lại sau cắt giữa theo giống', 'C3_head_box_per_image': None, 'C3_head_box_summary': 'Hộp bao vùng đầu: số ảnh có hộp, kích thước và phần bị cắt', 'C4_median_stats_by_breed': 'Trung vị các thống kê ảnh theo giống, mèo trainval', 'C4_breed_deviation_iqr': 'Độ lệch trung vị của từng giống so với toàn bộ ID, tính theo IQR', 'C4_median_stats_by_dataset': 'Trung vị các thống kê ảnh theo tập dữ liệu', 'C5_manual_review_counts': 'Kết quả duyệt bằng mắt 30 ảnh mỗi giống', 'C6_ood_similarity_image_stats': 'Mức giống mèo ID của từng tập OOD theo thống kê ảnh (KS D)', 'D1_breed_counts_train_val_test': 'Số ảnh mỗi giống mèo trong train, validation và test', 'D2_near_duplicate_test': None, 'D2_near_duplicate_counts_test': 'Số cặp trùng và gần trùng liên quan tới Oxford test', 'D3_test_quality_and_size': 'Chất lượng và kích thước ảnh Oxford test'}, []
OOD_GRID_CAPTIONS = {
    "dogs_probe": "Mười hai ảnh ngẫu nhiên (seed 0) của tập thăm dò, chó trainval",
    "dtd": "Mười hai ảnh ngẫu nhiên (seed 0) của DTD",
    "felibreed": "Mười hai ảnh ngẫu nhiên (seed 0) của FeliBreed",
}
TABLE_CSS = [
    {"selector": "caption", "props": "caption-side: top; text-align: left; font-weight: 600; font-size: 13px; color: black; padding: 6px 0;"},
    {"selector": "th", "props": "background: #eef2f7; color: black; font-weight: 600; text-align: center; padding: 4px 8px; border-bottom: 1px solid #555;"},
    {"selector": "td", "props": "color: black; padding: 3px 8px; text-align: right;"},
    {"selector": "tr:nth-child(even) td", "props": "background: #fafafa;"},
    {"selector": "", "props": "border-collapse: collapse; border-top: 1.5px solid black; border-bottom: 1.5px solid black; font-size: 12px; margin-bottom: 10px;"},
]


def caption_text(kind, name, text):
    number = 1 + sum(1 for k, *_ in CAPTION_LOG if k == kind)
    label = f"{kind} 3.{number}"
    CAPTION_LOG.append((kind, label, name, text))
    pd.DataFrame(CAPTION_LOG, columns=["Loại", "Số", "Tệp", "Tiêu đề"]).to_csv(TABLES / "captions.csv", index=False, encoding="utf-8-sig")
    return f"{label}. {text}"


def show_table(frame, name, index=False):
    """Hiện bảng có tiêu đề “Bảng 3.x” phía trên, định dạng số gọn; tệp CSV không đổi."""
    text = TABLE_CAPTIONS.get(name)
    shown = frame if index else frame.reset_index(drop=True)
    styler = shown.style.format(lambda v: f"{v:,.3f}".rstrip("0").rstrip(".").replace(",", " ") if isinstance(v, float) else v, na_rep="–")
    if not index:
        styler = styler.hide(axis="index")
    if text:
        styler = styler.set_caption(caption_text("Bảng", name, text))
    display(styler.set_table_styles(TABLE_CSS))


def read_rgb(path):
    # Giống hệt notebook: đọc header thật, ghép alpha lên nền xám 128 rồi chuyển RGB; không xoay theo EXIF
    with Image.open(path) as source:
        source.load()
        if source.mode in ["RGBA", "LA"] or "transparency" in source.info:
            rgba = source.convert("RGBA")
            return Image.alpha_composite(Image.new("RGBA", rgba.size, (128, 128, 128, 255)), rgba).convert("RGB")
        return source.convert("RGB")


def thumb(path, size=400):
    image = read_rgb(path)
    image.thumbnail((size, size))
    return image


def file_hash(path, algorithm):
    with open(path, "rb") as stream:
        return hashlib.file_digest(stream, algorithm).hexdigest()


def json_hash(data):
    return hashlib.sha256(json.dumps(data, sort_keys=True).encode()).hexdigest()


def image_axis(axis):
    axis.set_xticks([])
    axis.set_yticks([])
    axis.grid(False)
    for spine in axis.spines.values():
        spine.set_visible(False)


def evaluation_layout(width, height):
    # Cùng công thức với notebook (torchvision): kích thước sau Resize và góc trên-trái của CenterCrop
    short, long = sorted((width, height))
    resized_long = int(RESIZE * long / short)
    new_width, new_height = (RESIZE, resized_long) if width <= height else (resized_long, RESIZE)
    left = int(round((new_width - CROP) / 2.0))
    top = int(round((new_height - CROP) / 2.0))
    return new_width, new_height, left, top


print("Thư mục dữ liệu:", ROOT)

Thư mục dữ liệu: C:\Users\ASUS\Desktop\TGMT_Project\eda_data


In [2]:
# Kiểm MD5 ba archive gốc
rows = []
for name, expected in EXPECTED_MD5.items():
    path = ROOT / "archives" / name
    actual = file_hash(path, "md5") if path.exists() else None
    rows.append({"Archive": name, "MD5 yêu cầu": expected, "MD5 thực tế": actual or "(không có tệp)",
                 "Khớp": actual == expected if actual else None})
md5_table = pd.DataFrame(rows)
save_table(md5_table, "00_archive_md5")

# Manifest chia tập do notebook sinh ra: kiểm split_id và SHA-256 của fixed_split.csv
audit = json.loads((SPLITS / "data_audit.json").read_text(encoding="utf-8"))
assert audit["split_id"] == EXPECTED_SPLIT_ID, "split_id khác bản đã khóa"
assert file_hash(SPLITS / "fixed_split.csv", "sha256") == audit["fixed_split_sha256"], "fixed_split.csv khác data_audit.json"
print("split_id", audit["split_id"][:8] + "…", "| SHA-256 của fixed_split.csv khớp data_audit.json")

Archive,MD5 yêu cầu,MD5 thực tế,Khớp
images.tar.gz,5c4f3ee8e5d25df40f4fd59a7f44e54c,5c4f3ee8e5d25df40f4fd59a7f44e54c,True
annotations.tar.gz,95a8c909bbe2e81eed6a22bccdf3f68f,95a8c909bbe2e81eed6a22bccdf3f68f,True
dtd-r1.0.1.tar.gz,fff73e5086ae6bdbea199a49dfb8a4c1,fff73e5086ae6bdbea199a49dfb8a4c1,True


split_id 3b283d42… | SHA-256 của fixed_split.csv khớp data_audit.json


## 1. Nạp danh sách ảnh (Oxford chỉ trainval)

In [3]:
def read_list(path):
    rows = []
    for line in path.read_text().splitlines():
        if line.startswith("#") or not line.strip():
            continue
        image_id, class_id, species, _ = line.split()
        rows.append({"image_id": image_id, "class_id": int(class_id), "species": int(species), "breed": image_id.rsplit("_", 1)[0]})
    return pd.DataFrame(rows)


trainval = read_list(OXFORD / "annotations" / "trainval.txt")  # không đọc test.txt ở đây
split = pd.read_csv(SPLITS / "fixed_split.csv").set_index("image_id").split

cats = trainval[trainval.species == 1].copy()
cats["split"] = cats.image_id.map(split)  # chỉ tra các ảnh trainval; dòng test của manifest không được dùng
assert cats.split.isin(["train", "val"]).all(), "có ảnh trainval không nằm trong train/val của manifest"
cats["path"] = cats.image_id.map(lambda i: OXFORD / "images" / f"{i}.jpg")
cats["trimap_path"] = cats.image_id.map(lambda i: OXFORD / "annotations" / "trimaps" / f"{i}.png")
cats["xml_path"] = cats.image_id.map(lambda i: OXFORD / "annotations" / "xmls" / f"{i}.xml")
cats["has_valid_trimap"] = ~cats.image_id.isin(audit["empty_or_invalid_trimaps"])  # danh sách trimap rỗng/hỏng của notebook

dogs = trainval[trainval.species == 2].copy()
dogs["path"] = dogs.image_id.map(lambda i: OXFORD / "images" / f"{i}.jpg")

# Tập thăm dò: cùng cách lấy với notebook (RandomState 42, 20 ảnh mỗi giống chó trainval), kiểm bằng probe_id
rng = np.random.RandomState(PROBE_SEED)
probe = []
for breed in sorted(dogs.breed.unique()):
    names = sorted(dogs.loc[dogs.breed == breed, "image_id"])
    probe += sorted(rng.choice(names, PROBE_PER_BREED, replace=False).tolist())
assert json_hash(probe) == EXPECTED_PROBE_ID, "tập thăm dò khác danh sách đã cố định"
dogs["split"] = np.where(dogs.image_id.isin(probe), "probe", "trainval_khac")

dtd = pd.DataFrame([{"image_id": f"{p.parent.name}/{p.name}", "breed": p.parent.name, "path": p} for p in sorted(DTD.glob("*/*.jpg"))])
dtd["split"] = "far_ood"

feli_manifest = pd.read_csv(FELI / "manifest.csv")
feli = pd.DataFrame({"image_id": feli_manifest.file, "breed": feli_manifest.category,
                     "path": feli_manifest.file.map(lambda f: FELI / f), "split": "unseen_cat"})

frames = {"Mèo trainval (ID)": cats, "Chó trainval": dogs, "DTD": dtd, "FeliBreed": feli}
display(pd.DataFrame({name: {"Số ảnh": len(f), "Số lớp/nhóm": f.breed.nunique()} for name, f in frames.items()}).T)
display(cats.groupby("split").size().rename("Số ảnh mèo").to_frame().T)
print("Tập thăm dò:", int((dogs.split == "probe").sum()), "ảnh chó trainval, probe_id khớp")

,Số ảnh,Số lớp/nhóm
Mèo trainval (ID),1188,12
Chó trainval,2492,25
DTD,5640,47
FeliBreed,94,2


split,train,val
Số ảnh mèo,950,238


Tập thăm dò: 500 ảnh chó trainval, probe_id khớp


## 2. Đọc toàn bộ ảnh một lần (dùng cho A2, A3, B2, B3, C4, C6)

Với mỗi ảnh: SHA-256 nội dung tệp; PIL `verify()` rồi mở lại và `load()`; định dạng thật theo header; chế độ màu; cờ EXIF Orientation (0x0112); kích thước.
Sau đó đọc bằng `read_rgb` (cùng quy ước notebook) để tính:
- perceptual hash `imagehash.phash` (64 bit) trên ảnh gốc;
- độ sáng (trung bình kênh L), độ tương phản (độ lệch chuẩn L), độ nét (phương sai Laplacian), và thêm độ bão hòa trung bình (kênh S của HSV).
  Bốn thống kê này tính **trên ảnh đã Resize cạnh ngắn 256**, để các tập có độ phân giải rất khác nhau so sánh được và gần với đầu vào thật của mô hình.

Kết quả lưu ở `tables/image_inspection.csv`. Tệp đã có thì đọc lại, không tính lại.

In [4]:
def inspect(path):
    path = Path(path)
    row = {"path": str(path), "sha256": file_hash(path, "sha256")}
    try:
        with Image.open(path) as image:
            image.verify()  # phải gọi ngay sau khi mở
        with Image.open(path) as image:
            row.update(format=image.format, mode=image.mode, width=image.width, height=image.height,
                       exif_orientation=int(image.getexif().get(0x0112, 1)))
            image.load()
        row["error"] = ""
    except Exception as error:  # ghi lại, không bỏ qua
        row["error"] = f"{type(error).__name__}: {error}"
        return row
    rgb = read_rgb(path)
    row["phash"] = str(imagehash.phash(rgb))
    short = min(rgb.size)
    small = rgb.resize((round(rgb.width * RESIZE / short), round(rgb.height * RESIZE / short)), Image.BILINEAR)
    gray = np.asarray(small.convert("L"), dtype=np.float64)
    row["brightness"] = gray.mean()
    row["contrast"] = gray.std()
    row["sharpness"] = cv2.Laplacian(gray, cv2.CV_64F).var()
    row["saturation"] = np.asarray(small.convert("HSV"))[..., 1].mean()
    return row


cache = TABLES / "image_inspection.csv"
if not cache.exists():
    listing = pd.concat([f.assign(dataset=name)[["dataset", "image_id", "breed", "split", "path"]] for name, f in frames.items()],
                        ignore_index=True)
    listing["path"] = listing.path.astype(str)
    with ThreadPoolExecutor(max_workers=8) as pool:
        result = pd.DataFrame(list(pool.map(inspect, listing.path)))
    result = listing.merge(result, on="path")
    result["path"] = result.path.map(lambda p: Path(p).relative_to(ROOT).as_posix())
    result.to_csv(cache, index=False, encoding="utf-8-sig")
inspected = pd.read_csv(cache, dtype={"phash": str, "error": str}, encoding="utf-8-sig")
inspected["error"] = inspected.error.fillna("")
inspected["short_side"] = inspected[["width", "height"]].min(axis=1)
inspected["aspect"] = inspected.width / inspected.height
inspected["log_sharpness"] = np.log10(inspected.sharpness + 1)
inspected["dataset"] = pd.Categorical(inspected.dataset, DATASETS, ordered=True)
print(len(inspected), "ảnh đã đọc:", inspected.dataset.value_counts().reindex(DATASETS).to_dict())

9414 ảnh đã đọc: {'Mèo trainval (ID)': 1188, 'Chó trainval': 2492, 'DTD': 5640, 'FeliBreed': 94}


## A. Tổng quan và chất lượng
### A1. Mô tả bốn tập dữ liệu

In [5]:
a1 = pd.DataFrame([
    {"Tập": "Oxford-IIIT Pet, 12 giống mèo", "Vai trò": "ID (train / validation)",
     "Nguồn, cách thu thập": "Parkhi và cs., CVPR 2012; ảnh lấy từ Internet, nhãn giống, trimap và hộp bao vùng đầu do người gán",
     "Giấy phép": "CC BY-SA 4.0 (nhãn); bản quyền ảnh thuộc chủ ảnh",
     "Số ảnh (phạm vi EDA)": len(cats), "Số lớp": cats.breed.nunique(),
     "Ý nghĩa nhãn": "Giống của con mèo chính trong ảnh; trimap 1 = con vật, 2 = nền, 3 = biên; hộp bao vùng đầu (chỉ một phần ảnh)"},
    {"Tập": "Oxford-IIIT Pet, 25 giống chó", "Vai trò": "Tập thăm dò (500 ảnh chó trainval); near-OOD (chó test, sau khóa)",
     "Nguồn, cách thu thập": "Như trên", "Giấy phép": "Như trên",
     "Số ảnh (phạm vi EDA)": len(dogs), "Số lớp": dogs.breed.nunique(),
     "Ý nghĩa nhãn": "Giống chó; với CatBreedID chỉ mang nghĩa “không phải mèo thuộc 12 giống”"},
    {"Tập": "DTD r1.0.1", "Vai trò": "Far-OOD",
     "Nguồn, cách thu thập": "Cimpoi và cs., CVPR 2014; ảnh kết cấu từ Google và Flickr, nhãn kiểm qua Amazon Mechanical Turk",
     "Giấy phép": "Dùng cho nghiên cứu (theo trang DTD); bản quyền ảnh thuộc chủ ảnh",
     "Số ảnh (phạm vi EDA)": len(dtd), "Số lớp": dtd.breed.nunique(),
     "Ý nghĩa nhãn": "Tên kết cấu (47 nhóm × 120 ảnh); không có con vật"},
    {"Tập": "FeliBreed (Mixed Breed + Local)", "Vai trò": "unseen_cat: mèo ngoài 12 giống, chỉ mô tả",
     "Nguồn, cách thu thập": "Mahi, Arnab, Khan; Mendeley Data v2, 26/08/2025, DOI 10.17632/ng6tv57j5c.2; tác giả không mô tả cách thu thập và gán nhãn",
     "Giấy phép": "CC BY 4.0",
     "Số ảnh (phạm vi EDA)": len(feli), "Số lớp": feli.breed.nunique(),
     "Ý nghĩa nhãn": "Tên thư mục do tác giả đặt: “Mixed Breed” (mèo lai), “Local” (mèo ta); chưa được kiểm độc lập"},
])
save_table(a1, "A1_dataset_description")

Tập,Vai trò,"Nguồn, cách thu thập",Giấy phép,Số ảnh (phạm vi EDA),Số lớp,Ý nghĩa nhãn
"Oxford-IIIT Pet, 12 giống mèo",ID (train / validation),"Parkhi và cs., CVPR 2012; ảnh lấy từ Internet, nhãn giống, trimap và hộp bao vùng đầu do người gán",CC BY-SA 4.0 (nhãn); bản quyền ảnh thuộc chủ ảnh,1188,12,"Giống của con mèo chính trong ảnh; trimap 1 = con vật, 2 = nền, 3 = biên; hộp bao vùng đầu (chỉ một phần ảnh)"
"Oxford-IIIT Pet, 25 giống chó","Tập thăm dò (500 ảnh chó trainval); near-OOD (chó test, sau khóa)",Như trên,Như trên,2492,25,Giống chó; với CatBreedID chỉ mang nghĩa “không phải mèo thuộc 12 giống”
DTD r1.0.1,Far-OOD,"Cimpoi và cs., CVPR 2014; ảnh kết cấu từ Google và Flickr, nhãn kiểm qua Amazon Mechanical Turk",Dùng cho nghiên cứu (theo trang DTD); bản quyền ảnh thuộc chủ ảnh,5640,47,Tên kết cấu (47 nhóm × 120 ảnh); không có con vật
FeliBreed (Mixed Breed + Local),"unseen_cat: mèo ngoài 12 giống, chỉ mô tả","Mahi, Arnab, Khan; Mendeley Data v2, 26/08/2025, DOI 10.17632/ng6tv57j5c.2; tác giả không mô tả cách thu thập và gán nhãn",CC BY 4.0,94,2,"Tên thư mục do tác giả đặt: “Mixed Breed” (mèo lai), “Local” (mèo ta); chưa được kiểm độc lập"


### A2. Ảnh lỗi, định dạng thật, chế độ màu, cờ xoay EXIF

In [6]:
a2 = []
for name, group in inspected.groupby("dataset", observed=True):
    row = {"Tập": name, "Số ảnh": len(group), "Lỗi đọc": int((group.error != "").sum())}
    row.update({f"Định dạng {k}": int(v) for k, v in group.format.value_counts().items()})
    row.update({f"Màu {k}": int(v) for k, v in group["mode"].value_counts().items()})
    flagged = group.exif_orientation.fillna(1) != 1
    row["Có cờ xoay EXIF (≠ 1)"] = int(flagged.sum())
    row["Cờ xoay EXIF (%)"] = round(100 * flagged.mean(), 2)
    a2.append(row)
a2 = pd.DataFrame(a2).fillna(0)
a2 = a2.astype({c: int for c in a2.columns if c not in ["Tập", "Cờ xoay EXIF (%)"]})
save_table(a2, "A2_image_quality")

errors = inspected[inspected.error != ""][["dataset", "image_id", "error"]]
if len(errors):
    save_table(errors, "A2_read_errors")
else:
    display(Markdown("**Không có ảnh nào lỗi khi `verify()` và `load()`.**"))
mismatch = ~((inspected.format == "JPEG") & inspected.path.str.lower().str.endswith((".jpg", ".jpeg")))
odd = inspected[(inspected["mode"] != "RGB") | (inspected.exif_orientation.fillna(1) != 1) | mismatch]
odd = odd[["dataset", "image_id", "format", "mode", "exif_orientation", "width", "height"]]
save_table(odd, "A2_non_rgb_exif_or_format_mismatch")
print("FeliBreed: số ảnh đã được xoay theo EXIF và lưu lại khi chuẩn bị dataset:",
      int((feli_manifest.original_sha256 != feli_manifest.sha256).sum()))

Tập,Số ảnh,Lỗi đọc,Định dạng JPEG,Định dạng PNG,Màu RGB,Màu RGBA,Có cờ xoay EXIF (≠ 1),Cờ xoay EXIF (%)
Mèo trainval (ID),1188,0,1185,3,1186,2,2,0.17
Chó trainval,2492,0,2492,0,2492,0,0,0
DTD,5640,0,5640,0,5640,0,0,0
FeliBreed,94,0,94,0,94,0,0,0


**Không có ảnh nào lỗi khi `verify()` và `load()`.**

dataset,image_id,format,mode,exif_orientation,width,height
Mèo trainval (ID),Egyptian_Mau_10,JPEG,RGB,0,600,715
Mèo trainval (ID),Abyssinian_17,JPEG,RGB,0,275,411
Mèo trainval (ID),Egyptian_Mau_14,PNG,RGBA,1,582,800
Mèo trainval (ID),Egyptian_Mau_156,PNG,RGB,1,400,265
Mèo trainval (ID),Egyptian_Mau_186,PNG,RGBA,1,183,275


FeliBreed: số ảnh đã được xoay theo EXIF và lưu lại khi chuẩn bị dataset: 27


**Nhận xét A2.** *Quan sát:* 9.414 ảnh, 0 ảnh lỗi khi `verify()` và `load()`. Mèo trainval có 3 tệp PNG mang đuôi .jpg (Egyptian_Mau_14, _156, _186), 2 ảnh RGBA, 2 ảnh có thẻ EXIF Orientation = 0 (giá trị không hợp lệ, không phải lệnh xoay). Chó, DTD, FeliBreed: toàn JPEG RGB, không có cờ xoay (27 ảnh FeliBreed đã được xoay sẵn khi chuẩn bị dataset). *Suy đoán:* không ảnh nào bị đọc sai hướng; ảnh RGBA được `read_rgb` ghép nền xám 128 nên không ảnh hưởng. *Quyết định:* giữ tất cả.

### A3. Kích thước ảnh

In [7]:
rows = []
for name, group in inspected.groupby("dataset", observed=True):
    rows.append({
        "Tập": name, "Số ảnh": len(group),
        "Rộng trung vị": group.width.median(), "Cao trung vị": group.height.median(),
        "Cạnh ngắn nhỏ nhất": group.short_side.min(), "Cạnh ngắn trung vị": group.short_side.median(),
        "Cạnh ngắn lớn nhất": group.short_side.max(),
        "Tỷ lệ khung trung vị": round(group.aspect.median(), 3),
        "Tỷ lệ khung P5": round(group.aspect.quantile(0.05), 3), "Tỷ lệ khung P95": round(group.aspect.quantile(0.95), 3),
        "Cạnh ngắn < 224": int((group.short_side < CROP).sum()),
        "Cạnh ngắn < 224 (%)": round(100 * (group.short_side < CROP).mean(), 1),
        "Cạnh ngắn < 256": int((group.short_side < RESIZE).sum()),
        "Cạnh ngắn < 256 (%)": round(100 * (group.short_side < RESIZE).mean(), 1),
        "Ảnh ngang (%)": round(100 * (group.width > group.height).mean(), 1),
        "Ảnh dọc (%)": round(100 * (group.width < group.height).mean(), 1),
        "Ảnh vuông (%)": round(100 * (group.width == group.height).mean(), 1),
    })
a3 = pd.DataFrame(rows)
a3.to_csv(TABLES / "A3_image_size_summary.csv", index=False, encoding="utf-8-sig")
show_table(a3.set_index("Tập").T, "A3_image_size_summary", index=True)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), layout="constrained")
for axis, name in zip(axes, DATASETS):
    values = inspected.loc[inspected.dataset == name, "short_side"]
    bins = np.logspace(np.log10(values.min() * 0.9), np.log10(values.max() * 1.1), 30)
    axis.hist(values, bins=bins, color=COLORS[name], edgecolor="white", linewidth=0.5)
    axis.set_xscale("log")
    axis.axvline(CROP, color="black", linestyle=":", linewidth=1)
    axis.axvline(RESIZE, color="black", linestyle="--", linewidth=1)
    axis.set(xlabel="Cạnh ngắn (pixel, thang log)", ylabel="Số ảnh")
    axis.set_xlim(80, 4000)
    axis.set_xticks([100, 224, 500, 1000, 3000], ["100", "224", "500", "1000", "3000"])
    axis.minorticks_off()
    axis.set_title(f"{SHORT[name]} (n = {len(values)})", fontsize=10)
fig.legend(handles=[plt.Line2D([], [], color="black", linestyle=":", label="224 pixel"),
                    plt.Line2D([], [], color="black", linestyle="--", label="256 pixel")],
           loc="outside lower center", ncol=2)
save(fig, "A3_short_side_hist")

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), layout="constrained", sharex=True)
for axis, name in zip(axes, DATASETS):
    values = inspected.loc[inspected.dataset == name, "aspect"]
    axis.hist(values.clip(0.3, 3), bins=np.linspace(0.3, 3, 40), color=COLORS[name], edgecolor="white", linewidth=0.5)
    axis.axvline(1, color="black", linestyle="--", linewidth=1)
    axis.set(xlabel="Tỷ lệ khung rộng / cao", ylabel="Số ảnh")
    axis.set_title(SHORT[name], fontsize=10)
save(fig, "A3_aspect_ratio_hist")

Tập,Mèo trainval (ID),Chó trainval,DTD,FeliBreed
Số ảnh,1 188,2 492,5 640,94
Rộng trung vị,500,500,497,3 060
Cao trung vị,375,375,437,3 060
Cạnh ngắn nhỏ nhất,108,117,231,1 200
Cạnh ngắn trung vị,334,335,425,3 060
Cạnh ngắn lớn nhất,2 448,500,778,3 060
Tỷ lệ khung trung vị,1.333,1.333,1.071,1.037
Tỷ lệ khung P5,0.666,0.666,0.741,0.749
Tỷ lệ khung P95,1.506,1.502,1.508,1.338
Cạnh ngắn < 224,65,121,0,0


<p style='text-align:center'><b>Hình 3.1. Phân bố cạnh ngắn của ảnh trong bốn tập (thang log); nét chấm: 224 pixel, nét đứt: 256 pixel</b></p>

<p style='text-align:center'><b>Hình 3.2. Phân bố tỷ lệ khung rộng/cao của ảnh trong bốn tập; nét đứt: ảnh vuông</b></p>

**Nhận xét A3.** *Quan sát:* mèo ID và chó có kích thước gần như giống nhau (cạnh ngắn trung vị 334 và 335, tỷ lệ khung 4:3); 141 ảnh mèo (11,9%) và 386 ảnh chó (15,5%) có cạnh ngắn < 256, 65 ảnh mèo (5,5%) < 224. DTD: cạnh ngắn trung vị 425, 9% ảnh vuông. FeliBreed: ảnh điện thoại rất lớn (cạnh ngắn trung vị 3.060, nhỏ nhất 1.200). *Suy đoán:* ảnh nhỏ bị phóng to khi Resize 256 nên mờ hơn; FeliBreed bị thu nhỏ khoảng 12 lần, khác hẳn ID, nên khác biệt điểm OOD trên FeliBreed có thể một phần do nguồn ảnh chứ không chỉ do giống. *Quyết định:* giữ; ghi hạn chế cho FeliBreed.

### A4. Ảnh mẫu
6 ảnh mỗi giống mèo (trainval, seed 0); 12 ảnh ngẫu nhiên (seed 0) của mỗi tập OOD. Ảnh chó lấy từ 500 ảnh thăm dò.

In [8]:
fig, axes = plt.subplots(12, 6, figsize=(9.5, 19), layout="constrained")
for row_axes, breed in zip(axes, CAT_BREEDS):
    for axis, path in zip(row_axes, cats[cats.breed == breed].sample(6, random_state=SEED).path):
        axis.imshow(thumb(path))
        image_axis(axis)
    row_axes[0].set_ylabel(breed.replace("_", " "), rotation=0, ha="right", va="center", fontsize=9)
save(fig, "A4_cat_breeds_6_each")

for name, frame in [("dogs_probe", dogs[dogs.split == "probe"]), ("dtd", dtd), ("felibreed", feli)]:
    sample = frame.sample(12, random_state=SEED)
    fig, axes = plt.subplots(2, 6, figsize=(12, 4.4), layout="constrained")
    for axis, row in zip(axes.flat, sample.itertuples()):
        axis.imshow(thumb(row.path))
        axis.set_title(str(row.breed).replace("_", " "), fontsize=8)
        image_axis(axis)
    save(fig, f"A4_{name}_12_random")

<p style='text-align:center'><b>Hình 3.3. Sáu ảnh ngẫu nhiên (seed 0) của mỗi giống mèo, trainval</b></p>

<p style='text-align:center'><b>Hình 3.4. Mười hai ảnh ngẫu nhiên (seed 0) của tập thăm dò, chó trainval</b></p>

<p style='text-align:center'><b>Hình 3.5. Mười hai ảnh ngẫu nhiên (seed 0) của DTD</b></p>

<p style='text-align:center'><b>Hình 3.6. Mười hai ảnh ngẫu nhiên (seed 0) của FeliBreed</b></p>

**Nhận xét A4.** Lưới ảnh dùng để minh họa trong Chương 3. *Quan sát:* FeliBreed có nhiều mèo mặc áo, đeo kính, có người phía sau, chụp trong cùng một sự kiện; DTD không có con vật.

## B. Nhãn, trùng lặp, chia dữ liệu
### B1. Số ảnh mỗi giống mèo trong train và validation

In [9]:
b1 = pd.crosstab(cats.breed, cats.split).reindex(CAT_BREEDS)[["train", "val"]]
b1["trainval"] = b1.sum(axis=1)
b1["val / trainval (%)"] = (100 * b1.val / b1.trainval).round(1)
save_table(b1.reset_index().rename(columns={"breed": "Giống"}), "B1_breed_counts_train_val")
ratio = pd.DataFrame({
    "Phần": ["train", "val", "trainval"],
    "Lớp nhiều nhất": [f"{b1[c].idxmax()} ({b1[c].max()})" for c in ["train", "val", "trainval"]],
    "Lớp ít nhất": [f"{b1[c].idxmin()} ({b1[c].min()})" for c in ["train", "val", "trainval"]],
    "Tỷ lệ lớn nhất / nhỏ nhất": [round(b1[c].max() / b1[c].min(), 3) for c in ["train", "val", "trainval"]],
})
save_table(ratio, "B1_imbalance_ratio")

fig, axis = plt.subplots(figsize=(9, 4.2))
x = np.arange(len(CAT_BREEDS))
axis.bar(x, b1.train, color=SPLIT_COLORS["train"], label="Train")
axis.bar(x, b1.val, bottom=b1.train, color=SPLIT_COLORS["val"], label="Validation")
for i, (t, v) in enumerate(zip(b1.train, b1.val)):
    axis.text(i, t / 2, str(t), ha="center", va="center", color="white", fontsize=8)
    axis.text(i, t + v / 2, str(v), ha="center", va="center", color="black", fontsize=8)
axis.set_xticks(x, [b.replace("_", " ") for b in CAT_BREEDS], rotation=35, ha="right")
axis.set(ylabel="Số ảnh")
axis.grid(axis="x", visible=False)
axis.legend(loc="upper center", bbox_to_anchor=(0.5, 1.12), ncol=2)
save(fig, "B1_breed_counts_train_val")

Giống,train,val,trainval,val / trainval (%)
Abyssinian,80,20,100,20
Bengal,80,20,100,20
Birman,80,20,100,20
Bombay,77,19,96,19.8
British_Shorthair,80,20,100,20
Egyptian_Mau,74,19,93,20.4
Maine_Coon,80,20,100,20
Persian,80,20,100,20
Ragdoll,80,20,100,20
Russian_Blue,80,20,100,20


Phần,Lớp nhiều nhất,Lớp ít nhất,Tỷ lệ lớn nhất / nhỏ nhất
train,Abyssinian (80),Egyptian_Mau (74),1.081
val,Abyssinian (20),Bombay (19),1.053
trainval,Abyssinian (100),Egyptian_Mau (93),1.075


<p style='text-align:center'><b>Hình 3.7. Số ảnh mỗi giống mèo trong train và validation</b></p>

**Nhận xét B1.** *Quan sát:* train 950, validation 238; mỗi giống 74–80 ảnh train và 19–20 ảnh validation; tỷ lệ lớn nhất / nhỏ nhất 1,08 (train). *Suy đoán:* gần như cân bằng, không cần trọng số lớp. *Quyết định:* giữ, không xử lý.

### B2. Trùng hoàn toàn và gần trùng trong mèo trainval, và giữa train với validation
- Trùng hoàn toàn: cùng SHA-256 nội dung tệp.
- Gần trùng: `imagehash.phash` 64 bit, khoảng cách Hamming. Ngưỡng đề xuất ≤ 4; báo thêm 0, 2, 6, 8 để thấy độ nhạy.
- `tables/near_duplicates_trainval.csv` liệt kê mọi cặp có Hamming ≤ 8 (cột `hamming` để lọc theo ngưỡng).

In [10]:
def hash_bits(hashes):
    values = np.array([int(h, 16) for h in hashes], dtype=np.uint64)
    return np.unpackbits(values.view(np.uint8).reshape(-1, 8), axis=1).astype(bool)


def hamming_pairs(left, right, same, limit=8):
    a, b = hash_bits(left.phash), hash_bits(right.phash)
    distance = np.stack([(a[k][None, :] != b).sum(axis=1) for k in range(len(a))])
    i, j = np.nonzero(distance <= limit)
    if same:
        keep = i < j
        i, j = i[keep], j[keep]
    L, R = left.iloc[i].reset_index(drop=True), right.iloc[j].reset_index(drop=True)
    pairs = pd.DataFrame({"image_a": L.image_id, "dataset_a": L.dataset.astype(str), "breed_a": L.breed, "split_a": L.split,
                          "image_b": R.image_id, "dataset_b": R.dataset.astype(str), "breed_b": R.breed, "split_b": R.split,
                          "hamming": distance[i, j], "same_sha256": L.sha256.values == R.sha256.values})
    return pairs.sort_values(["hamming", "image_a"]).reset_index(drop=True)


def threshold_table(pairs, label, cross_split=False):
    rows = []
    for t in [0, 2, 4, 6, 8]:
        chosen = pairs[pairs.hamming <= t]
        row = {"Phạm vi": label, "Hamming ≤": t, "Số cặp": len(chosen)}
        if cross_split:
            row["Cặp train–val"] = int((chosen.split_a != chosen.split_b).sum())
        row["Cặp khác giống/nhóm"] = int((chosen.breed_a != chosen.breed_b).sum())
        row["Số ảnh liên quan"] = len(set(chosen.image_a) | set(chosen.image_b))
        rows.append(row)
    return pd.DataFrame(rows)


PATH_OF = dict(zip(inspected.image_id, inspected.path.map(lambda p: ROOT / p)))


def pair_grid(pairs, name, per_band=3, bands=((0, 2), (3, 4), (5, 6), (7, 8))):
    chosen = [row for low, high in bands for _, row in pairs[pairs.hamming.between(low, high)].head(per_band).iterrows()]
    if not chosen:
        print("Không có cặp nào có Hamming ≤ 8, không vẽ", name)
        return
    fig, axes = plt.subplots(len(chosen), 2, figsize=(5.2, 2.4 * len(chosen)), layout="constrained", squeeze=False)
    for (left, right), row in zip(axes, chosen):
        for axis, image_id, part in [(left, row.image_a, row.split_a), (right, row.image_b, row.split_b)]:
            axis.imshow(thumb(PATH_OF[image_id]))
            image_axis(axis)
            axis.set_title(f"{image_id} ({part})", fontsize=7)
        left.set_ylabel(f"Hamming {row.hamming}", fontsize=8)
    save(fig, name)


cat_rows = inspected[inspected.dataset == "Mèo trainval (ID)"].reset_index(drop=True)
exact = cat_rows[cat_rows.duplicated("sha256", keep=False)].sort_values("sha256")
print("Mèo trainval: số nhóm trùng SHA-256 =", exact.sha256.nunique(), "| số ảnh =", len(exact))
if len(exact):
    display(exact[["image_id", "breed", "split", "sha256"]])

near_cats = hamming_pairs(cat_rows, cat_rows, same=True)
save_table(near_cats, "near_duplicates_trainval")
b2 = threshold_table(near_cats, "Mèo trainval", cross_split=True)
save_table(b2, "B2_near_duplicate_threshold_counts")
pair_grid(near_cats, "B2_near_duplicate_examples")

Mèo trainval: số nhóm trùng SHA-256 = 0 | số ảnh = 0


image_a,dataset_a,breed_a,split_a,image_b,dataset_b,breed_b,split_b,hamming,same_sha256
Abyssinian_11,Mèo trainval (ID),Abyssinian,val,Abyssinian_177,Mèo trainval (ID),Abyssinian,train,0,False
Bombay_107,Mèo trainval (ID),Bombay,val,Bombay_132,Mèo trainval (ID),Bombay,train,0,False
Bombay_107,Mèo trainval (ID),Bombay,val,Bombay_19,Mèo trainval (ID),Bombay,train,0,False
Bombay_113,Mèo trainval (ID),Bombay,train,Bombay_157,Mèo trainval (ID),Bombay,train,0,False
Bombay_132,Mèo trainval (ID),Bombay,train,Bombay_19,Mèo trainval (ID),Bombay,train,0,False
Bombay_136,Mèo trainval (ID),Bombay,val,Bombay_150,Mèo trainval (ID),Bombay,val,0,False
Abyssinian_122,Mèo trainval (ID),Abyssinian,train,Abyssinian_172,Mèo trainval (ID),Abyssinian,val,2,False
Bombay_114,Mèo trainval (ID),Bombay,train,Bombay_139,Mèo trainval (ID),Bombay,train,2,False
Ragdoll_160,Mèo trainval (ID),Ragdoll,train,Ragdoll_161,Mèo trainval (ID),Ragdoll,train,6,False


Phạm vi,Hamming ≤,Số cặp,Cặp train–val,Cặp khác giống/nhóm,Số ảnh liên quan
Mèo trainval,0,6,3,0,9
Mèo trainval,2,8,4,0,13
Mèo trainval,4,8,4,0,13
Mèo trainval,6,9,4,0,15
Mèo trainval,8,9,4,0,15


<p style='text-align:center'><b>Hình 3.8. Các cặp ảnh gần trùng trong mèo trainval theo khoảng cách Hamming của phash</b></p>

**Nhận xét B2.** *Quan sát:* 0 cặp trùng SHA-256, nhưng 6 cặp có Hamming = 0 (cùng một ảnh lưu lại với mã hóa khác) và 8 cặp ≤ 4; số cặp không đổi từ ngưỡng 2 đến 4, chỉ thêm 1 cặp ở ngưỡng 6 (Ragdoll_160/161, hai ảnh chụp liên tiếp). Xem tay: cả 9 cặp đều là cùng con mèo, cùng cảnh, cùng giống. **4 cặp vắt qua train–validation**: Abyssinian_11 (val) – Abyssinian_177, Abyssinian_172 (val) – Abyssinian_122, Bombay_107 (val) – Bombay_132 và Bombay_19. *Suy đoán:* 3 ảnh validation có bản sao trong train nên độ chính xác validation có thể bị thổi nhẹ (tối đa 3/238 ≈ 1,3 điểm). *Quyết định:* không chia lại (split đã khóa); ghi hạn chế và gửi Hậu danh sách để cân nhắc báo cáo thêm kết quả validation bỏ 3 ảnh này.

### B3. Gần trùng giữa FeliBreed và Oxford trainval (mèo lẫn chó), và trong nội bộ FeliBreed

In [11]:
oxford_rows = inspected[inspected.dataset.isin(["Mèo trainval (ID)", "Chó trainval"])].reset_index(drop=True)
feli_rows = inspected[inspected.dataset == "FeliBreed"].reset_index(drop=True)
feli_vs_oxford = hamming_pairs(feli_rows, oxford_rows, same=False)
feli_internal = hamming_pairs(feli_rows, feli_rows, same=True)
print("FeliBreed trùng SHA-256 với Oxford trainval:", int(feli_rows.sha256.isin(oxford_rows.sha256).sum()))
print("FeliBreed trùng SHA-256 nội bộ:", int(feli_rows.sha256.duplicated().sum()))
b3 = pd.concat([threshold_table(feli_vs_oxford, "FeliBreed – Oxford trainval"), threshold_table(feli_internal, "FeliBreed nội bộ")])
save_table(b3, "B3_felibreed_near_duplicate_counts")
save_table(pd.concat([feli_vs_oxford.assign(scope="FeliBreed – Oxford trainval"), feli_internal.assign(scope="FeliBreed nội bộ")]),
           "near_duplicates_felibreed")
pair_grid(feli_internal, "B3_felibreed_internal_examples")
if len(feli_vs_oxford):
    pair_grid(feli_vs_oxford, "B3_felibreed_vs_oxford_examples")

FeliBreed trùng SHA-256 với Oxford trainval: 0
FeliBreed trùng SHA-256 nội bộ: 0


Phạm vi,Hamming ≤,Số cặp,Cặp khác giống/nhóm,Số ảnh liên quan
FeliBreed – Oxford trainval,0,0,0,0
FeliBreed – Oxford trainval,2,0,0,0
FeliBreed – Oxford trainval,4,0,0,0
FeliBreed – Oxford trainval,6,0,0,0
FeliBreed – Oxford trainval,8,0,0,0
FeliBreed nội bộ,0,0,0,0
FeliBreed nội bộ,2,0,0,0
FeliBreed nội bộ,4,0,0,0
FeliBreed nội bộ,6,2,0,4
FeliBreed nội bộ,8,4,0,8


image_a,dataset_a,breed_a,split_a,image_b,dataset_b,breed_b,split_b,hamming,same_sha256,scope
FeliBreed/Local/36.jpg,FeliBreed,Local,unseen_cat,FeliBreed/Local/47.jpg,FeliBreed,Local,unseen_cat,6,False,FeliBreed nội bộ
FeliBreed/Local/38.jpg,FeliBreed,Local,unseen_cat,FeliBreed/Local/43.jpg,FeliBreed,Local,unseen_cat,6,False,FeliBreed nội bộ
FeliBreed/Local/41.jpg,FeliBreed,Local,unseen_cat,FeliBreed/Local/52.jpg,FeliBreed,Local,unseen_cat,8,False,FeliBreed nội bộ
FeliBreed/Local/57.jpg,FeliBreed,Local,unseen_cat,FeliBreed/Local/60.jpg,FeliBreed,Local,unseen_cat,8,False,FeliBreed nội bộ


<p style='text-align:center'><b>Hình 3.9. Các cặp ảnh gần trùng trong nội bộ FeliBreed</b></p>

**Nhận xét B3.** *Quan sát:* FeliBreed không trùng SHA-256 và không có cặp nào Hamming ≤ 8 với Oxford trainval (mèo lẫn chó). Trong nội bộ FeliBreed: 0 cặp ≤ 4, nhưng có 2 cặp ở ngưỡng 6 và 4 cặp ở ngưỡng 8; xem tay đều là **ảnh chụp liên tiếp cùng một con mèo** (Local 36/47, 38/43, 41/52, 57/60). phash chỉ bắt được ảnh gần như cùng khung hình, nên số ảnh cùng cá thể có thể còn nhiều hơn. *Suy đoán:* 94 ảnh không phải 94 mẫu độc lập; khoảng tin cậy nên tính theo cá thể. *Quyết định:* không cần loại ảnh nào khỏi FeliBreed vì không trùng Oxford; ghi hạn chế, đề xuất gom ảnh theo cá thể bằng mắt.

### B4. Tên giống trong FeliBreed so với 12 giống Oxford
Số ảnh đếm theo thư mục trong tệp zip Mendeley v2. Trang dataset ghi Mixed 41, Local 59, khác với số ảnh thật trong zip (37, 57).

In [12]:
b4 = pd.DataFrame([
    {"Thư mục FeliBreed": "Persian", "Số ảnh trong zip": 158, "Tương ứng Oxford": "Persian (cùng giống)", "Kết luận": "Loại: thuộc 12 giống ID"},
    {"Thư mục FeliBreed": "Bengal Cat", "Số ảnh trong zip": 14, "Tương ứng Oxford": "Bengal (cùng giống)", "Kết luận": "Loại: thuộc 12 giống ID"},
    {"Thư mục FeliBreed": "Tabby Cat", "Số ảnh trong zip": 28,
     "Tương ứng Oxford": "Không phải tên giống: “tabby” là kiểu lông vằn, gặp ở Bengal, Egyptian Mau, Maine Coon, British Shorthair…",
     "Kết luận": "Loại: có thể là mèo thuộc giống ID"},
    {"Thư mục FeliBreed": "Mixed Breed", "Số ảnh trong zip": 37, "Tương ứng Oxford": "Không có tên tương ứng",
     "Kết luận": "Giữ (unseen_cat); nhãn chưa được kiểm độc lập"},
    {"Thư mục FeliBreed": "Local", "Số ảnh trong zip": 57, "Tương ứng Oxford": "Không có tên tương ứng (mèo nhà, giống địa phương)",
     "Kết luận": "Giữ (unseen_cat); nhãn chưa được kiểm độc lập"},
])
kept = sorted(feli.breed.unique())
assert not {k.lower() for k in kept} & {b.replace("_", " ").lower() for b in CAT_BREEDS}
save_table(b4, "B4_felibreed_breed_names")
print("Thư mục có trong dataset dùng:", kept, "| tổng", len(feli), "ảnh")

Thư mục FeliBreed,Số ảnh trong zip,Tương ứng Oxford,Kết luận
Persian,158,Persian (cùng giống),Loại: thuộc 12 giống ID
Bengal Cat,14,Bengal (cùng giống),Loại: thuộc 12 giống ID
Tabby Cat,28,"Không phải tên giống: “tabby” là kiểu lông vằn, gặp ở Bengal, Egyptian Mau, Maine Coon, British Shorthair…",Loại: có thể là mèo thuộc giống ID
Mixed Breed,37,Không có tên tương ứng,Giữ (unseen_cat); nhãn chưa được kiểm độc lập
Local,57,"Không có tên tương ứng (mèo nhà, giống địa phương)",Giữ (unseen_cat); nhãn chưa được kiểm độc lập


Thư mục có trong dataset dùng: ['Local', 'Mixed Breed'] | tổng 94 ảnh


**Nhận xét B4.** *Quan sát:* Persian và Bengal Cat trùng tên với giống ID; “Tabby” là kiểu lông, không phải giống. *Quyết định:* chỉ giữ Mixed Breed (37 ảnh) và Local (57 ảnh); tác giả không mô tả cách gán nhãn hai thư mục này, nên ghi hạn chế.

## C. Nội dung ảnh, liên quan tới tiền xử lý
### C1. Kích thước và vị trí con vật theo trimap (mèo trainval có trimap hợp lệ)
Tiền cảnh là pixel nhãn 1 của trimap. Độ lệch tâm là khoảng cách từ tâm tiền cảnh tới tâm ảnh, tính theo tỷ lệ chiều rộng và chiều cao (0 = đúng tâm, 0,5 = ở mép).

In [13]:
def trimap_stats(row):
    with Image.open(row.trimap_path) as source:
        mask = np.array(source) == 1
    ys, xs = np.nonzero(mask)
    h, w = mask.shape
    return {"image_id": row.image_id, "fg_fraction": mask.mean(), "cx": (xs.mean() + 0.5) / w, "cy": (ys.mean() + 0.5) / h}


valid = cats[cats.has_valid_trimap]
with ThreadPoolExecutor(max_workers=8) as pool:
    c1 = pd.DataFrame(list(pool.map(trimap_stats, [r for _, r in valid.iterrows()])))
c1 = valid[["image_id", "breed", "split"]].merge(c1, on="image_id")
c1["offset"] = np.hypot(c1.cx - 0.5, c1.cy - 0.5)
save_table(c1, "C1_trimap_foreground_per_image")
g = c1.groupby("breed")
c1_table = pd.DataFrame({
    "Số ảnh": g.size(), "Tiền cảnh trung vị": g.fg_fraction.median(), "Tiền cảnh P5": g.fg_fraction.quantile(0.05),
    "Tiền cảnh < 10%": g.fg_fraction.apply(lambda s: int((s < 0.1).sum())),
    "Lệch tâm trung vị": g.offset.median(), "Lệch tâm P95": g.offset.quantile(0.95),
    "Lệch tâm > 0,15": g.offset.apply(lambda s: int((s > 0.15).sum())),
}).reindex(CAT_BREEDS)
c1_table.loc["Toàn bộ"] = [len(c1), c1.fg_fraction.median(), c1.fg_fraction.quantile(0.05), int((c1.fg_fraction < 0.1).sum()),
                           c1.offset.median(), c1.offset.quantile(0.95), int((c1.offset > 0.15).sum())]
c1_table = c1_table.round(3)
save_table(c1_table.reset_index(), "C1_trimap_foreground_summary")
print("Trimap không dùng được (trainval):", list(cats.loc[~cats.has_valid_trimap, "image_id"]))

fig, axis = plt.subplots(figsize=(6.5, 3.4))
axis.hist(c1.fg_fraction, bins=np.linspace(0, 1, 41), color=COLORS["Mèo trainval (ID)"], edgecolor="white", linewidth=0.5)
axis.axvline(c1.fg_fraction.median(), color="black", linestyle="--", linewidth=1, label=f"trung vị {c1.fg_fraction.median():.2f}")
axis.set(xlabel="Tỷ lệ pixel con vật trên ảnh (trimap = 1)", ylabel="Số ảnh")
axis.legend()
save(fig, "C1_foreground_fraction_hist")

fig, axes = plt.subplots(3, 5, figsize=(12, 7.6), layout="constrained")
edges = np.linspace(0, 1, 11)
for axis, breed in zip(axes.flat, CAT_BREEDS + ["Toàn bộ"]):
    part = c1 if breed == "Toàn bộ" else c1[c1.breed == breed]
    counts, *_ = np.histogram2d(part.cy, part.cx, bins=[edges, edges])
    image = axis.imshow(counts / len(part), extent=[0, 1, 1, 0], cmap="Greys", vmin=0, vmax=0.35)
    axis.plot(0.5, 0.5, marker="+", color="#d43d3d", markersize=9)
    axis.set_title(f"{breed.replace('_', ' ')} (n = {len(part)})", fontsize=9)
    axis.set_xticks([0, 0.5, 1])
    axis.set_yticks([0, 0.5, 1])
    axis.grid(False)
for axis in axes.flat[13:]:
    axis.axis("off")
fig.colorbar(image, ax=axes, shrink=0.6, label="Tỷ lệ ảnh có tâm tiền cảnh trong ô")
fig.supxlabel("Hoành độ tâm tiền cảnh / chiều rộng ảnh")
fig.supylabel("Tung độ tâm tiền cảnh / chiều cao ảnh")
save(fig, "C1_foreground_centre_heatmap")

breed,Số ảnh,Tiền cảnh trung vị,Tiền cảnh P5,Tiền cảnh < 10%,Lệch tâm trung vị,Lệch tâm P95,"Lệch tâm > 0,15"
Abyssinian,100,0.241,0.1,5,0.107,0.215,29
Bengal,100,0.268,0.088,6,0.105,0.219,28
Birman,100,0.355,0.126,3,0.127,0.211,34
Bombay,96,0.305,0.131,2,0.111,0.205,22
British_Shorthair,100,0.323,0.124,1,0.11,0.23,30
Egyptian_Mau,90,0.222,0.112,2,0.092,0.217,16
Maine_Coon,100,0.332,0.118,2,0.101,0.211,28
Persian,100,0.451,0.15,1,0.089,0.176,14
Ragdoll,100,0.379,0.132,4,0.105,0.216,19
Russian_Blue,100,0.288,0.085,7,0.126,0.226,36


Trimap không dùng được (trainval): ['Egyptian_Mau_162', 'Egyptian_Mau_165', 'Egyptian_Mau_196']


<p style='text-align:center'><b>Hình 3.10. Phân bố tỷ lệ diện tích con vật trên ảnh theo trimap, mèo trainval</b></p>

<p style='text-align:center'><b>Hình 3.11. Vị trí tâm con vật trong ảnh theo từng giống, mèo trainval; dấu + là tâm ảnh</b></p>

**Nhận xét C1.** *Quan sát:* 1.185 ảnh có trimap hợp lệ (3 ảnh Egyptian Mau trainval có trimap rỗng). Con vật chiếm trung vị 30% diện tích ảnh (Persian 45%, Egyptian Mau 22%); 48 ảnh có tiền cảnh < 10%. Tâm con vật lệch trung vị 0,11 so với tâm ảnh; 314 ảnh (26,5%) lệch > 0,15; bản đồ nhiệt tập trung quanh tâm, hơi lệch xuống dưới. *Suy đoán:* cắt giữa hợp lý với đa số ảnh, nhưng ảnh có con vật nhỏ hoặc lệch tâm dễ mất thông tin. *Quyết định:* giữ CenterCrop; kiểm bằng C2 và D7.

### C2. Tiền cảnh còn lại sau Resize 256 → CenterCrop 224
Bàn giao yêu cầu dùng `crop_retention_trainval.csv` do notebook sinh ra. Nếu tệp này đặt ở `eda_data/splits/`, cell dưới dùng nó và so sánh với bản tính lại.
Nếu chưa có, cell tính lại **đúng công thức của hàm `eda_crop_retention` trong `notebooks/kaggle_runner.ipynb` (commit c45e3bc trên `main`)**: trimap resize cạnh ngắn 256 bằng NEAREST, cắt giữa 224, tỷ lệ pixel nhãn 1 còn lại. Với ảnh PIL, `torchvision` gọi đúng `PIL.Image.resize(NEAREST)` và dùng cùng công thức làm tròn góc cắt, nên hai cách cho cùng kết quả.

In [14]:
def retained(path):
    with Image.open(path) as source:
        new_width, new_height, left, top = evaluation_layout(*source.size)
        full = np.array(source.resize((new_width, new_height), Image.NEAREST))
    crop = full[top:top + CROP, left:left + CROP]
    return float((crop == 1).sum() / max(1, (full == 1).sum()))


with ThreadPoolExecutor(max_workers=8) as pool:
    c2 = valid[["image_id", "breed", "split"]].assign(foreground_retained=list(pool.map(retained, valid.trimap_path)))
official = SPLITS / "crop_retention_trainval.csv"
if official.exists():
    from_notebook = pd.read_csv(official)
    compare = c2.merge(from_notebook[["image_id", "foreground_retained"]], on="image_id", suffixes=("_eda", ""))
    print("Dùng tệp của notebook. Chênh lệch lớn nhất so với bản tính lại:",
          float((compare.foreground_retained_eda - compare.foreground_retained).abs().max()))
    c2 = compare[["image_id", "breed", "split", "foreground_retained"]]
    C2_SOURCE = "crop_retention_trainval.csv của notebook"
else:
    C2_SOURCE = "tính lại theo đúng công thức của notebook; chưa có tệp do notebook sinh ra"
print("Nguồn số liệu C2:", C2_SOURCE)
save_table(c2, "crop_retention_trainval")
v = c2.foreground_retained
c2_table = pd.Series({
    "Số ảnh trainval có trimap": len(v), "Trung bình": round(v.mean(), 4), "Trung vị": round(v.median(), 4),
    "Phân vị 5%": round(v.quantile(0.05), 4), "Nhỏ nhất": round(v.min(), 4),
    "Giữ < 80% tiền cảnh (ảnh)": int((v < 0.8).sum()), "Giữ < 80% (%)": round(100 * (v < 0.8).mean(), 1),
    "Giữ < 90% (%)": round(100 * (v < 0.9).mean(), 1), "Giữ đủ 100% (%)": round(100 * (v >= 0.9999).mean(), 1),
}).to_frame("Giá trị")
save_table(c2_table.reset_index().rename(columns={"index": "Chỉ số"}), "C2_crop_retention_summary")
by_breed = c2.groupby("breed").foreground_retained.agg(
    trung_vi="median", P5=lambda s: s.quantile(0.05), so_anh_duoi_80=lambda s: int((s < 0.8).sum())).reindex(CAT_BREEDS).round(3)
save_table(by_breed.reset_index(), "C2_crop_retention_by_breed")

fig, axis = plt.subplots(figsize=(6.5, 3.4))
axis.hist(v, bins=np.linspace(np.floor(v.min() * 20) / 20, 1, 40), color=COLORS["Mèo trainval (ID)"], edgecolor="white", linewidth=0.5)
axis.axvline(0.8, color="black", linestyle="--", linewidth=1, label="80%")
axis.set(xlabel="Tỷ lệ pixel con vật còn lại sau CenterCrop 224", ylabel="Số ảnh")
axis.legend()
save(fig, "C2_crop_retention_hist")

worst = c2.nsmallest(8, "foreground_retained").merge(cats[["image_id", "path"]], on="image_id")
fig, axes = plt.subplots(2, 4, figsize=(12, 6.2), layout="constrained")
for axis, row in zip(axes.flat, worst.itertuples()):
    image = read_rgb(row.path)
    w, h = image.size
    new_width, new_height, left, top = evaluation_layout(w, h)
    sx, sy = w / new_width, h / new_height
    axis.imshow(image)
    axis.add_patch(Rectangle((left * sx, top * sy), CROP * sx, CROP * sy, fill=False, edgecolor="#d43d3d", linewidth=2))
    axis.set_title(f"{row.image_id}: giữ {row.foreground_retained:.0%}", fontsize=8)
    image_axis(axis)
save(fig, "C2_crop_retention_worst")

Nguồn số liệu C2: tính lại theo đúng công thức của notebook; chưa có tệp do notebook sinh ra


Chỉ số,Giá trị
Số ảnh trainval có trimap,1 185
Trung bình,0.863
Trung vị,0.891
Phân vị 5%,0.645
Nhỏ nhất,0.419
Giữ < 80% tiền cảnh (ảnh),347
Giữ < 80% (%),29.3
Giữ < 90% (%),52.2
Giữ đủ 100% (%),8.6


breed,trung_vi,P5,so_anh_duoi_80
Abyssinian,0.911,0.652,19
Bengal,0.887,0.659,25
Birman,0.852,0.662,38
Bombay,0.912,0.721,19
British_Shorthair,0.889,0.676,34
Egyptian_Mau,0.944,0.697,14
Maine_Coon,0.879,0.636,33
Persian,0.825,0.649,44
Ragdoll,0.855,0.618,35
Russian_Blue,0.87,0.619,39


<p style='text-align:center'><b>Hình 3.12. Tỷ lệ pixel con vật còn lại sau Resize 256 → CenterCrop 224, mèo trainval</b></p>

<p style='text-align:center'><b>Hình 3.13. Tám ảnh mất nhiều tiền cảnh nhất khi cắt giữa; khung đỏ là phần còn lại</b></p>

**Nhận xét C2.** *Quan sát* (nguồn: tính lại đúng công thức của notebook; sẽ đối chiếu khi có tệp `crop_retention_trainval.csv` của Hậu): trung vị 89% tiền cảnh được giữ; chỉ 8,6% ảnh giữ đủ 100%; **347 ảnh (29,3%) giữ < 80%**; ảnh tệ nhất giữ 42% (Abyssinian_175). Persian bị cắt nhiều nhất (trung vị 83%, 44 ảnh < 80%). *Suy đoán:* với ảnh 4:3, CenterCrop bỏ hai bên ảnh, làm mất thân, đuôi, đôi khi cả đầu. *Quyết định:* căn cứ cho D7 (so sánh letterbox trên validation).

### C3. Hộp bao vùng đầu (`annotations/xmls`)
Phần hộp còn lại sau cắt giữa: diện tích hộp sau khi đổi sang hệ 224×224 và cắt theo khung (như `transform_box` của notebook), chia cho diện tích hộp trước khi cắt.

In [15]:
def head_box(path):
    box = ET.parse(path).getroot().find("object/bndbox")
    return [float(box.find(k).text) for k in ["xmin", "ymin", "xmax", "ymax"]]


def box_retained(box, w, h):
    new_width, new_height, left, top = evaluation_layout(w, h)
    sx, sy = new_width / w, new_height / h
    x0, y0, x1, y1 = box[0] * sx - left, box[1] * sy - top, box[2] * sx - left, box[3] * sy - top
    full = (x1 - x0) * (y1 - y0)
    kept = max(0, min(x1, CROP) - max(x0, 0)) * max(0, min(y1, CROP) - max(y0, 0))
    return kept / full if full > 0 else np.nan


sizes = inspected.set_index("image_id")[["width", "height"]]
rows = []
for r in cats.itertuples():
    row = {"image_id": r.image_id, "breed": r.breed, "split": r.split, "has_box": Path(r.xml_path).exists()}
    if row["has_box"]:
        box = head_box(r.xml_path)
        w, h = sizes.loc[r.image_id]
        row.update(box_area_rel=(box[2] - box[0]) * (box[3] - box[1]) / (w * h), box_w_rel=(box[2] - box[0]) / w,
                   box_h_rel=(box[3] - box[1]) / h, box_retained=box_retained(box, w, h))
    rows.append(row)
c3 = pd.DataFrame(rows)
save_table(c3, "C3_head_box_per_image")
g = c3.groupby("breed")
c3_table = pd.DataFrame({
    "Số ảnh": g.size(), "Có hộp đầu": g.has_box.sum().astype(int),
    "Diện tích hộp / ảnh (trung vị)": g.box_area_rel.median(), "Rộng hộp / rộng ảnh (trung vị)": g.box_w_rel.median(),
    "Hộp bị cắt một phần": g.box_retained.apply(lambda s: int((s < 0.999).sum())),
    "Hộp còn < 80%": g.box_retained.apply(lambda s: int((s < 0.8).sum())),
}).reindex(CAT_BREEDS)
c3_table.loc["Toàn bộ"] = [len(c3), int(c3.has_box.sum()), c3.box_area_rel.median(), c3.box_w_rel.median(),
                           int((c3.box_retained < 0.999).sum()), int((c3.box_retained < 0.8).sum())]
c3_table = c3_table.round(3)
save_table(c3_table.reset_index(), "C3_head_box_summary")
display(c3.groupby("split").has_box.agg(["size", "sum"]).rename(columns={"size": "Số ảnh", "sum": "Có hộp đầu"}))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), layout="constrained")
boxed = c3[c3.has_box]
axes[0].hist(boxed.box_area_rel, bins=40, color=COLORS["Mèo trainval (ID)"], edgecolor="white", linewidth=0.5)
axes[0].set(xlabel="Diện tích hộp đầu / diện tích ảnh", ylabel="Số ảnh")
axes[1].hist(boxed.box_retained, bins=np.linspace(0, 1, 41), color=COLORS["Mèo trainval (ID)"], edgecolor="white", linewidth=0.5)
axes[1].set(xlabel="Phần hộp đầu còn lại sau CenterCrop 224", ylabel="Số ảnh")
save(fig, "C3_head_box_hist")

breed,Số ảnh,Có hộp đầu,Diện tích hộp / ảnh (trung vị),Rộng hộp / rộng ảnh (trung vị),Hộp bị cắt một phần,Hộp còn < 80%
Abyssinian,100,99,0.122,0.356,65,27
Bengal,100,98,0.106,0.278,62,31
Birman,100,100,0.171,0.404,55,14
Bombay,96,96,0.152,0.352,67,23
British_Shorthair,100,100,0.212,0.417,48,19
Egyptian_Mau,93,90,0.072,0.253,67,42
Maine_Coon,100,100,0.129,0.347,51,25
Persian,100,100,0.18,0.457,67,34
Ragdoll,100,99,0.182,0.38,64,29
Russian_Blue,100,100,0.17,0.362,67,32


,Số ảnh,Có hộp đầu
split,,
train,950,943
val,238,238


<p style='text-align:center'><b>Hình 3.14. Kích thước tương đối của hộp đầu (trái) và phần hộp đầu còn lại sau cắt giữa (phải)</b></p>

**Nhận xét C3.** *Quan sát:* 1.181/1.188 ảnh trainval có hộp đầu (train 943/950, validation 238/238). Hộp đầu chiếm trung vị 15% diện tích ảnh (Egyptian Mau 7%). **741 ảnh (62,7%) có hộp đầu bị cắt một phần** sau CenterCrop, 333 ảnh còn < 80% hộp. *Suy đoán:* phần bị cắt thường là mép tai; mô hình có thể mất đặc điểm tai (Abyssinian, Sphynx). *Quyết định:* nối với phân tích `head_crop` trên `val_head`.

### C4. Độ sáng, độ tương phản, độ nét theo giống và theo tập
Tính trên ảnh đã Resize cạnh ngắn 256 (mục 2). Độ nét là phương sai Laplacian, vẽ theo log10. Độ bão hòa là thống kê bổ sung, không có trong bàn giao.

In [16]:
METRICS = {"brightness": "Độ sáng (trung bình L)", "contrast": "Độ tương phản (độ lệch chuẩn L)",
           "log_sharpness": "Độ nét (log10 phương sai Laplacian)", "saturation": "Độ bão hòa (trung bình S)"}
ok = inspected[inspected.error == ""]
id_rows = ok[ok.dataset == "Mèo trainval (ID)"]

fig, axes = plt.subplots(2, 2, figsize=(13, 7.5), layout="constrained")
for axis, (column, label) in zip(axes.flat, METRICS.items()):
    axis.boxplot([id_rows.loc[id_rows.breed == b, column] for b in CAT_BREEDS], widths=0.6, patch_artist=True,
                 boxprops={"facecolor": "#b7d3f6", "edgecolor": "#2a6fdb"}, medianprops={"color": "black"}, flierprops={"markersize": 2})
    axis.set_xticks(range(1, 13), [b.replace("_", " ") for b in CAT_BREEDS], rotation=35, ha="right", fontsize=8)
    axis.set_ylabel(label)
    axis.grid(axis="x", visible=False)
save(fig, "C4_image_stats_by_breed")

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), layout="constrained")
for axis, (column, label) in zip(axes, METRICS.items()):
    parts = axis.boxplot([ok.loc[ok.dataset == d, column] for d in DATASETS], widths=0.6, patch_artist=True,
                         medianprops={"color": "black"}, flierprops={"markersize": 2})
    for patch, d in zip(parts["boxes"], DATASETS):
        patch.set_facecolor(COLORS[d])
        patch.set_alpha(0.55)
    axis.set_xticks(range(1, 5), [SHORT[d] for d in DATASETS])
    axis.set_ylabel(label)
    axis.grid(axis="x", visible=False)
save(fig, "C4_image_stats_by_dataset")

c4_breed = id_rows.groupby("breed")[list(METRICS)].median().reindex(CAT_BREEDS)
c4_breed.loc["Toàn bộ ID"] = id_rows[list(METRICS)].median()
display(c4_breed.round(2))
save_table(c4_breed.round(3).reset_index(), "C4_median_stats_by_breed")
iqr = id_rows[list(METRICS)].quantile(0.75) - id_rows[list(METRICS)].quantile(0.25)
deviation = ((c4_breed.drop(index="Toàn bộ ID") - id_rows[list(METRICS)].median()) / iqr).round(2)
print("Độ lệch trung vị của từng giống so với toàn bộ ID, tính theo IQR của ID (|giá trị| ≥ 0,5 coi là lệch rõ):")
save_table(deviation.reset_index(), "C4_breed_deviation_iqr")
c4_dataset = ok.groupby("dataset", observed=True)[list(METRICS)].median().round(2)
save_table(c4_dataset.reset_index(), "C4_median_stats_by_dataset")

<p style='text-align:center'><b>Hình 3.15. Độ sáng, độ tương phản, độ nét và độ bão hòa theo giống, mèo trainval</b></p>

<p style='text-align:center'><b>Hình 3.16. Độ sáng, độ tương phản, độ nét và độ bão hòa theo tập dữ liệu</b></p>

,brightness,contrast,log_sharpness,saturation
breed,,,,
Abyssinian,117.46,51.52,2.43,84.33
Bengal,115.70,52.63,2.52,75.39
Birman,113.65,57.90,2.39,57.05
Bombay,102.62,69.10,2.34,56.78
British_Shorthair,121.67,57.58,2.62,56.95
Egyptian_Mau,113.15,49.78,2.54,49.32
Maine_Coon,107.99,51.77,2.62,76.62
Persian,121.25,53.68,2.42,74.68
Ragdoll,120.92,56.61,2.33,62.31


breed,brightness,contrast,log_sharpness,saturation
Abyssinian,117.463,51.524,2.433,84.331
Bengal,115.696,52.631,2.524,75.393
Birman,113.649,57.903,2.392,57.05
Bombay,102.621,69.103,2.335,56.777
British_Shorthair,121.671,57.582,2.623,56.945
Egyptian_Mau,113.15,49.784,2.542,49.316
Maine_Coon,107.989,51.774,2.615,76.622
Persian,121.248,53.677,2.418,74.683
Ragdoll,120.919,56.606,2.328,62.308
Russian_Blue,123.062,58.794,2.611,47.36


Độ lệch trung vị của từng giống so với toàn bộ ID, tính theo IQR của ID (|giá trị| ≥ 0,5 coi là lệch rõ):


breed,brightness,contrast,log_sharpness,saturation
Abyssinian,0.04,-0.24,-0.13,0.29
Bengal,-0,-0.18,0.06,0.13
Birman,-0.05,0.08,-0.22,-0.2
Bombay,-0.33,0.64,-0.34,-0.2
British_Shorthair,0.15,0.06,0.28,-0.2
Egyptian_Mau,-0.06,-0.33,0.1,-0.34
Maine_Coon,-0.19,-0.23,0.26,0.15
Persian,0.14,-0.13,-0.16,0.11
Ragdoll,0.13,0.01,-0.35,-0.11
Russian_Blue,0.18,0.12,0.25,-0.37


dataset,brightness,contrast,log_sharpness,saturation
Mèo trainval (ID),115.72,56.33,2.49,68.29
Chó trainval,114.38,55.83,2.67,68.93
DTD,122.8,40.3,2.89,70.64
FeliBreed,101.4,60.29,2.48,87.71


**Nhận xét C4.** *Quan sát:* giữa các giống, chỉ Bombay lệch rõ (độ tương phản +0,64 IQR, do mèo đen trên nền sáng); các giống khác lệch < 0,4 IQR. Giữa các tập: chó gần như trùng mèo ID; DTD có độ tương phản thấp hơn (40 so với 56) và nét hơn; FeliBreed tối hơn (101 so với 116) và bão hòa cao hơn nhiều (88 so với 68). *Suy đoán:* mô hình có thể dùng màu và độ sáng làm dấu hiệu phụ cho Bombay; FeliBreed khác ID cả về chất lượng ảnh. *Quyết định:* ghi hạn chế, không xử lý.

### C5. Thiên lệch: duyệt bằng mắt 30 ảnh mỗi giống (seed 0)
Cell dưới vẽ 12 tờ ảnh `figures/C5_review_<giống>.png` (mỗi ảnh có số thứ tự) và tạo `tables/manual_review.csv`, mỗi ảnh một dòng.
**Người duyệt điền tay** các cột: `nen` (`trong_nha` / `ngoai_troi` / `khong_ro`), `co_nguoi`, `nhieu_con_vat`, `chu_hoac_khung` (0 hoặc 1), `ghi_chu`.
Tệp đã có thì không bị ghi đè. Điền xong thì chạy lại cell để có bảng đếm theo giống.

In [17]:
review_path = TABLES / "manual_review.csv"
review_sample = pd.concat([cats[cats.breed == b].sample(30, random_state=SEED) for b in CAT_BREEDS])
for breed in CAT_BREEDS:
    fig, axes = plt.subplots(5, 6, figsize=(12, 10.5), layout="constrained")
    for k, (axis, row) in enumerate(zip(axes.flat, review_sample[review_sample.breed == breed].itertuples()), 1):
        axis.imshow(thumb(row.path, 500))
        axis.set_title(f"{k}. {row.image_id}", fontsize=7)
        image_axis(axis)
    fig.savefig(FIGURES / f"C5_review_{breed}.png", dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
if not review_path.exists():
    template = review_sample[["breed", "image_id", "split"]].copy()
    template.insert(1, "stt", template.groupby("breed").cumcount() + 1)
    for column in ["nen", "co_nguoi", "nhieu_con_vat", "chu_hoac_khung", "ghi_chu"]:
        template[column] = ""
    template.to_csv(review_path, index=False, encoding="utf-8-sig")
    print("Đã tạo mẫu tables/manual_review.csv, chờ người duyệt điền")
review = pd.read_csv(review_path, dtype=str, encoding="utf-8-sig").fillna("")
filled = review[review.nen != ""].copy()
print(f"Đã điền {len(filled)}/{len(review)} dòng")
if len(filled):
    flags = ["co_nguoi", "nhieu_con_vat", "chu_hoac_khung"]
    filled[flags] = filled[flags].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)
    c5 = pd.concat([filled.groupby("breed").size().rename("Đã duyệt"), pd.crosstab(filled.breed, filled.nen),
                    filled.groupby("breed")[flags].sum()], axis=1).reindex(CAT_BREEDS)
    c5.loc["Toàn bộ"] = c5.sum()
    display(c5)
    save_table(c5.reset_index(), "C5_manual_review_counts")

Đã điền 360/360 dòng


,Đã duyệt,khong_ro,ngoai_troi,trong_nha,co_nguoi,nhieu_con_vat,chu_hoac_khung
breed,,,,,,,
Abyssinian,30,8,0,22,1,0,10
Bengal,30,3,4,23,0,1,2
Birman,30,4,7,19,0,0,2
Bombay,30,10,4,16,0,0,10
British_Shorthair,30,3,4,23,0,0,4
Egyptian_Mau,30,14,3,13,0,0,9
Maine_Coon,30,3,6,21,1,0,5
Persian,30,5,4,21,1,0,5
Ragdoll,30,3,3,24,0,0,2


breed,Đã duyệt,khong_ro,ngoai_troi,trong_nha,co_nguoi,nhieu_con_vat,chu_hoac_khung
Abyssinian,30,8,0,22,1,0,10
Bengal,30,3,4,23,0,1,2
Birman,30,4,7,19,0,0,2
Bombay,30,10,4,16,0,0,10
British_Shorthair,30,3,4,23,0,0,4
Egyptian_Mau,30,14,3,13,0,0,9
Maine_Coon,30,3,6,21,1,0,5
Persian,30,5,4,21,1,0,5
Ragdoll,30,3,3,24,0,0,2
Russian_Blue,30,2,4,24,0,0,3


**Nhận xét C5.** Duyệt bằng mắt 360 ảnh (30 ảnh mỗi giống, seed 0) ngày 16/11/2026: Claude (trợ lý AI) duyệt và đánh dấu 19 ảnh chưa chắc, xem lại ở độ phân giải gốc; Võ Tấn Tài duyệt các nhận xét và quy ước. Quy ước: “chữ hoặc khung” chỉ tính chữ, watermark, logo, khung được chèn vào ảnh, không tính chữ có sẵn trên đồ vật; phông trơn chụp studio ghi “không rõ” nền. *Quan sát:* 255 ảnh (70,8%) trong nhà, 45 (12,5%) ngoài trời, 60 (16,7%) không rõ nền; **63 ảnh (17,5%) có chữ ký, watermark hoặc khung**, nhiều nhất ở Abyssinian và Bombay (10/30), Egyptian Mau (9/30); Egyptian Mau có 14/30 ảnh phông studio, Bombay 10/30; chỉ 3 ảnh có người (bàn tay, cánh tay) và 1 ảnh có hai con mèo. Ghi chú thêm: Bengal_185 trông giống mèo báo hoang hơn mèo Bengal nhà. *Suy đoán:* chữ ký của cùng một nhiếp ảnh gia lặp lại trong một giống (“©Chanan” ở Egyptian Mau và Bombay, “Helmi” ở Abyssinian, Bombay, Egyptian Mau; “SallyW” ở Abyssinian) và phông studio tập trung ở Egyptian Mau, Bombay là dấu hiệu phụ mô hình có thể học thay cho đặc điểm con mèo. *Quyết định:* không xử lý; ghi hạn chế, đối chiếu với Grad-CAM xem mô hình có chú ý vào vùng chữ ký hay không.

### C6. Mức giống ID của từng tập OOD, chỉ bằng thống kê ảnh
So phân bố từng thống kê của mỗi tập OOD với mèo ID bằng thống kê Kolmogorov–Smirnov hai mẫu, D (0 = hai phân bố trùng nhau, 1 = tách hẳn). Không dùng đặc trưng hay điểm của mô hình.

In [18]:
COLUMNS = {"short_side": "Cạnh ngắn", "aspect": "Tỷ lệ khung", "brightness": "Độ sáng", "contrast": "Độ tương phản",
           "log_sharpness": "Độ nét (log10)", "saturation": "Độ bão hòa"}
rows = []
for d in DATASETS:
    part = ok[ok.dataset == d]
    row = {"Tập": d, "Số ảnh": len(part)}
    for column, label in COLUMNS.items():
        row[f"{label}: trung vị"] = round(part[column].median(), 2)
    for column, label in COLUMNS.items():
        row[f"{label}: KS D với ID"] = np.nan if d == DATASETS[0] else round(stats.ks_2samp(part[column], id_rows[column]).statistic, 3)
    rows.append(row)
c6 = pd.DataFrame(rows)
c6["KS D trung bình (6 thống kê)"] = c6[[c for c in c6.columns if c.endswith("KS D với ID")]].mean(axis=1).round(3)
c6["Có con vật"] = ["mèo", "chó", "không", "mèo"]
c6["Cùng nguồn ảnh với ID"] = ["có", "có (Oxford)", "không", "không"]
display(c6.set_index("Tập").T)
c6.to_csv(TABLES / "C6_ood_similarity_image_stats.csv", index=False, encoding="utf-8-sig")
show_table(c6.set_index("Tập").T, "C6_ood_similarity_image_stats", index=True)

Tập,Mèo trainval (ID),Chó trainval,DTD,FeliBreed
Số ảnh,1188,2492,5640,94
Cạnh ngắn: trung vị,334.0,335.0,425.0,3060.0
Tỷ lệ khung: trung vị,1.33,1.33,1.07,1.04
Độ sáng: trung vị,115.72,114.38,122.8,101.4
Độ tương phản: trung vị,56.33,55.83,40.3,60.29
Độ nét (log10): trung vị,2.49,2.67,2.89,2.48
Độ bão hòa: trung vị,68.29,68.93,70.64,87.71
Cạnh ngắn: KS D với ID,NaN,0.063,0.47,0.994
Tỷ lệ khung: KS D với ID,NaN,0.052,0.275,0.272
Độ sáng: KS D với ID,NaN,0.062,0.115,0.396


Tập,Mèo trainval (ID),Chó trainval,DTD,FeliBreed
Số ảnh,1188,2492,5640,94
Cạnh ngắn: trung vị,334,335,425,3 060
Tỷ lệ khung: trung vị,1.33,1.33,1.07,1.04
Độ sáng: trung vị,115.72,114.38,122.8,101.4
Độ tương phản: trung vị,56.33,55.83,40.3,60.29
Độ nét (log10): trung vị,2.49,2.67,2.89,2.48
Độ bão hòa: trung vị,68.29,68.93,70.64,87.71
Cạnh ngắn: KS D với ID,–,0.063,0.47,0.994
Tỷ lệ khung: KS D với ID,–,0.052,0.275,0.272
Độ sáng: KS D với ID,–,0.062,0.115,0.396


**Nhận xét C6.** *Quan sát:* KS D trung bình so với mèo ID: chó 0,08, DTD 0,29, FeliBreed 0,44. Chó gần ID ở mọi thống kê (cùng nguồn Oxford); DTD khác ở độ tương phản, độ nét, kích thước; FeliBreed khác nhiều nhất, chủ yếu ở kích thước ảnh (D = 0,99), độ bão hòa và độ sáng. *Suy đoán:* gọi chó là near-OOD hợp lý cả về nội dung lẫn thống kê ảnh; DTD là far-OOD theo nội dung. FeliBreed cùng loài với ID nhưng khác nguồn ảnh rõ, nên kết quả trên FeliBreed lẫn hai yếu tố “giống khác” và “nguồn ảnh khác”. *Quyết định:* ghi rõ điều này khi trình bày kết quả unseen_cat.

## D. Sau khi khóa (chỉ chạy khi `ALLOW_TEST = True`)
D1: số ảnh mỗi giống mèo trong test, bảng tổng train / validation / test. D2: trùng và gần trùng giữa test và trainval (mèo lẫn chó), giữa FeliBreed và Oxford test. D3: lặp lại A2, A3 cho Oxford test.
Nếu D2 có cặp trùng: **không xóa** ảnh, không chạy lại; ghi số lượng vào Chương 3 như một hạn chế.

In [19]:
if not ALLOW_TEST:
    print("Mục D đang tắt (ALLOW_TEST = False): chưa mở test.txt, chưa đọc ảnh test.")
else:
    test = read_list(OXFORD / "annotations" / "test.txt")
    test["path"] = test.image_id.map(lambda i: str(OXFORD / "images" / f"{i}.jpg"))
    test["dataset"] = np.where(test.species == 1, "Mèo test", "Chó test")
    test["split"] = "test"
    with ThreadPoolExecutor(max_workers=8) as pool:
        test_rows = test.merge(pd.DataFrame(list(pool.map(inspect, test.path))), on="path")
    test_rows["error"] = test_rows.error.fillna("")
    test_rows["short_side"] = test_rows[["width", "height"]].min(axis=1)
    test_rows["aspect"] = test_rows.width / test_rows.height
    test_rows.assign(path=test_rows.path.map(lambda p: Path(p).relative_to(ROOT).as_posix())).to_csv(
        TABLES / "image_inspection_test.csv", index=False, encoding="utf-8-sig")
    # D1
    d1 = pd.concat([b1[["train", "val"]], test[test.species == 1].breed.value_counts().rename("test")], axis=1).reindex(CAT_BREEDS)
    d1["Tổng"] = d1.sum(axis=1)
    d1.loc["Tổng"] = d1.sum()
    display(d1)
    save_table(d1.reset_index(), "D1_breed_counts_train_val_test")
    # D2
    good = test_rows[test_rows.error == ""].reset_index(drop=True)
    d2_pairs = pd.concat([hamming_pairs(good, oxford_rows, same=False).assign(scope="Oxford test – Oxford trainval"),
                          hamming_pairs(feli_rows, good, same=False).assign(scope="FeliBreed – Oxford test")])
    save_table(d2_pairs, "D2_near_duplicates_test")
    d2 = pd.DataFrame({scope: {f"Hamming ≤ {t}": int((p.hamming <= t).sum()) for t in [0, 2, 4, 6, 8]} | {"Trùng SHA-256": int(p.same_sha256.sum())}
                       for scope, p in d2_pairs.groupby("scope")}).T
    display(d2)
    save_table(d2.reset_index(), "D2_near_duplicate_counts_test")
    # D3
    d3 = pd.DataFrame([{"Tập": name, "Số ảnh": len(gr), "Lỗi đọc": int((gr.error != "").sum()),
                        "Không phải RGB": int((gr["mode"] != "RGB").sum()), "Định dạng khác JPEG": int((gr.format != "JPEG").sum()),
                        "Cờ xoay EXIF": int((gr.exif_orientation.fillna(1) != 1).sum()),
                        "Cạnh ngắn < 224": int((gr.short_side < CROP).sum()), "Cạnh ngắn < 256": int((gr.short_side < RESIZE).sum()),
                        "Cạnh ngắn trung vị": gr.short_side.median(), "Tỷ lệ khung trung vị": round(gr.aspect.median(), 3)}
                       for name, gr in test_rows.groupby("dataset")])
    display(d3)
    save_table(d3, "D3_test_quality_and_size")

Mục D đang tắt (ALLOW_TEST = False): chưa mở test.txt, chưa đọc ảnh test.


## Bảng kết luận

| Phát hiện (số liệu) | Ảnh hưởng dự kiến | Đã xử lý thế nào / thực nghiệm nào kiểm tra | Ghi vào hạn chế? |
|---|---|---|---|
| 347/1.185 ảnh (29,3%) giữ < 80% tiền cảnh sau CenterCrop; chỉ 8,6% giữ đủ | Mất thân, đuôi, đôi khi cả đầu | D7: so sánh letterbox trên validation | Có |
| 741/1.181 ảnh (62,7%) bị cắt một phần hộp đầu; 333 ảnh còn < 80% | Mất đặc điểm tai và đầu | Phân tích `head_crop` trên `val_head` | Có |
| 4 cặp gần trùng vắt qua train–validation (3 ảnh validation có bản sao trong train); tổng 8 cặp Hamming ≤ 4 | Độ chính xác validation có thể bị thổi tối đa 1,3 điểm | Không chia lại; Hậu quyết định có báo cáo thêm kết quả validation bỏ 3 ảnh hay không | Có |
| 3 ảnh PNG mang đuôi .jpg, 2 ảnh RGBA | Không đáng kể | `read_rgb` đọc header thật, ghép alpha lên nền xám 128 | Không |
| 2 ảnh có thẻ EXIF Orientation = 0, không ảnh nào cần xoay; 27 ảnh FeliBreed đã được xoay sẵn | Không | Notebook không xoay theo EXIF | Không (nêu một câu) |
| 141 ảnh mèo (11,9%) có cạnh ngắn < 256, 65 ảnh < 224 | Ảnh bị phóng to, mờ hơn | Không xử lý | Có |
| 3 trimap trainval rỗng (Egyptian_Mau_162, _165, _196) | Thiếu ảnh khi tính IoU và xóa nền | Notebook loại khỏi các phân tích dùng trimap | Có |
| Lớp gần cân bằng: lớn nhất / nhỏ nhất 1,08 | Không | Không cần trọng số lớp | Không |
| FeliBreed: cạnh ngắn trung vị 3.060 (ID 334), bão hòa 88 (ID 68), KS D trung bình 0,44 | Kết quả unseen_cat lẫn hiệu ứng nguồn ảnh | Chỉ mô tả, không đặt giả thuyết (QĐ6) | Có |
| FeliBreed: không trùng Oxford; ít nhất 4 cặp ảnh chụp liên tiếp cùng một con mèo; nhãn chưa được kiểm | 94 ảnh không độc lập | Đề xuất gom theo cá thể và duyệt bằng mắt | Có |
| Bombay lệch độ tương phản +0,64 IQR so với ID | Mô hình có thể dựa vào màu lông đen | Grad-CAM (đã có trong notebook chính) | Không |
| Chó gần ID nhất (KS D 0,08), DTD 0,29 | Ủng hộ cách gọi near-OOD / far-OOD | Không cần | Không |
| 63/360 ảnh (17,5%) có chữ ký, watermark hoặc khung; nhiều ở Abyssinian, Bombay (10/30), Egyptian Mau (9/30); Egyptian Mau 14/30 phông studio | Mô hình có thể học chữ ký, phông thay cho con mèo | Đối chiếu Grad-CAM | Có |